# UAE Rental Price Prediction — 05 Classification Models and Evaluation

**Goal:** predict the existing `Rent_category` label (Low / Medium / High). Compare Decision Tree with Random Forest on the same train/test split.

**Input:** `UAE_properties_features.csv` from Notebook 03. **`Rent` is excluded**, along with all Rent-derived features, because the classification label was created from Rent. This keeps training fair.

We use stratified 5-fold cross-validation, tune Decision Tree settings, inspect accuracy, macro-F1, per-class recall, and confusion matrix, then save the selected model.

**Run the cells from top to bottom.** Calculated results appear only after you run the notebook using your dataset.

## 1. Load the data

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

paths = [
    Path('../data/UAE_properties_features.csv'),
    Path('data/UAE_properties_features.csv'),
    Path('UAE_properties_features.csv'),
    Path('/content/UAE_properties_features.csv'),
]
source_path = next((p for p in paths if p.is_file()), None)
if source_path is None:
    raise FileNotFoundError('Run Notebook 03 first, then make UAE_properties_features.csv available here.')
df = pd.read_csv(source_path, low_memory=False)
print('Loaded:', source_path)
print('Dataset size:', df.shape)
print('Missing cells:', int(df.isna().sum().sum()))
assert {'Rent','Rent_category'}.issubset(df.columns)
display(df.head(3))

## 2. Define the inputs and target

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# No Rent-derived columns may be used to predict the category created from Rent.
leaky = ['Rent','Rent_category','Rent_per_sqft']
X = df.drop(columns=leaky,errors='ignore').select_dtypes(include='number').copy()
y = df['Rent_category'].astype(str)
valid = y.isin(['Low','Medium','High']) & X.notna().all(axis=1)
X,y = X.loc[valid],y.loc[valid]
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.20,stratify=y,random_state=42)
print('Inputs:',len(X.columns),'Training:',len(X_train),'Testing:',len(X_test))
print('Class balance (entire set):')
display(y.value_counts().to_frame('Count'))

## 3. Compare baseline, Decision Tree and Random Forest

A Decision Tree uses understandable splitting rules. Random Forest combines many trees to improve generalization. A majority-class baseline shows how much better the learned models are.

In [ ]:
models = {
    'Majority baseline':DummyClassifier(strategy='most_frequent'),
    'Decision Tree':DecisionTreeClassifier(max_depth=10,min_samples_leaf=50,random_state=42),
    'Random Forest':RandomForestClassifier(n_estimators=30,max_depth=12,min_samples_leaf=30,random_state=42,n_jobs=1)
}
cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
cv_rows=[]
for name,model in models.items():
    result = cross_validate(model,X_train,y_train,cv=cv,
                            scoring={'accuracy':'accuracy','f1_macro':'f1_macro'},n_jobs=1)
    cv_rows.append({'Model':name,'CV_Accuracy':result['test_accuracy'].mean(),
                    'CV_Macro_F1':result['test_f1_macro'].mean(),
                    'CV_F1_Std':result['test_f1_macro'].std()})
cv_table = pd.DataFrame(cv_rows).sort_values('CV_Macro_F1',ascending=False)
display(cv_table.round(3))

## 4. Tune the Decision Tree (training data only)

In [ ]:
search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid={'max_depth':[6,10], 'min_samples_leaf':[30,100]},
    scoring='f1_macro',cv=3,n_jobs=1,refit=True
)
search.fit(X_train,y_train)
print('Best tree settings:',search.best_params_)
print('Tuning CV macro-F1:',round(search.best_score_,3))
models['Tuned Decision Tree'] = search.best_estimator_

## 5. Final metrics on the held-out test set

Accuracy shows overall correct predictions. Macro-F1 gives equal weight to Low, Medium and High, which is important even if class sizes differ.

In [ ]:
results=[]
for name,model in models.items():
    model.fit(X_train,y_train)
    pred=model.predict(X_test)
    results.append({'Model':name,'Test_Accuracy':accuracy_score(y_test,pred),
                    'Test_Macro_F1':f1_score(y_test,pred,average='macro')})
performance = pd.DataFrame(results).sort_values('Test_Macro_F1',ascending=False)
display(performance.round(3))

# Select by cross-validation on training data, not by test accuracy.
winner_name = cv_table.iloc[0]['Model']
if winner_name=='Decision Tree':
    tree_cv = cv_table.loc[cv_table.Model.eq('Decision Tree'),'CV_Macro_F1'].iloc[0]
    if search.best_score_ > tree_cv:
        winner_name='Tuned Decision Tree'
best_model = models[winner_name]
best_model.fit(X_train,y_train)
y_pred = best_model.predict(X_test)
print('Model selected using training CV:',winner_name)
print(classification_report(y_test,y_pred,labels=['Low','Medium','High'],zero_division=0))

## 6. Confusion matrix (Low, Medium, High)

The diagonal shows correct predictions. Off-diagonal numbers show classification mistakes that could lead to wrong rental-market recommendations.

In [ ]:
import matplotlib.pyplot as plt
labels = ['Low','Medium','High']
cm = confusion_matrix(y_test,y_pred,labels=labels)
display(pd.DataFrame(cm,index=['Actual '+x for x in labels],columns=['Predicted '+x for x in labels]))
ConfusionMatrixDisplay(confusion_matrix=cm,display_labels=labels).plot(cmap='Blues')
plt.title('Rent Category Confusion Matrix')
plt.tight_layout()
plt.show()

## 7. Model reliability and feature importance

Small variation between validation folds indicates more stable results; always verify test recall for the High category too. Feature importance is not proof of causation.

In [ ]:
from sklearn.metrics import recall_score
print('CV macro-F1 summary:')
display(cv_table[['Model','CV_Macro_F1','CV_F1_Std']].round(3))
print('High class recall:',round(recall_score(y_test,y_pred,labels=['High'],average='macro',zero_division=0),3))
if hasattr(best_model,'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_,index=X.columns).sort_values(ascending=False).head(10)
    display(imp.rename('Importance').to_frame())
    plt.figure(figsize=(8,4))
    plt.barh(imp.index[::-1],imp.values[::-1])
    plt.xlabel('Importance')
    plt.title('Top Predictors for Rental Category')
    plt.tight_layout()
    plt.show()
print('Business decision: NO-GO for automatic rental categorization until a business-approved')
print('minimum macro-F1 and recall per category are defined and fresh listings are tested.')

## 8. Save the trained model

In [ ]:
import joblib
project_root = source_path.parent.parent if source_path.parent.name=='data' else source_path.parent
model_dir = project_root/'models'
model_dir.mkdir(parents=True,exist_ok=True)
model_path = model_dir/'classification_model.joblib'
joblib.dump({'model':best_model,'feature_columns':X.columns.tolist(),
             'target':'Rent_category','model_name':winner_name},model_path)
print('Saved:',model_path)